In [26]:
"""
EDI X12 Spec Scraper — stedi.com
==================================
Generic scraper for any X12 transaction set and release.

Usage:
    pip install requests beautifulsoup4

    # As a script:
    python scrape_315_spec.py [path/to/file.edi] [--edi 315] [--release 004010]

    # As a module:
    from scrape_315_spec import fetch_transaction_set, fetch_segment_spec, extract_unique_segments
    tx  = fetch_transaction_set(edi="315", release="004010")
    seg = fetch_segment_spec("B4", release="004010")
"""

import csv, json, re, sys, time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Optional

import requests
from bs4 import BeautifulSoup

# ── Config ────────────────────────────────────────────────────────────────────
BASE    = "https://www.stedi.com"
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; edi-spec-scraper/1.0)"}
SLEEP   = 0.8   # polite delay between requests

def _normalize(edi: "str | int", release: "str | int") -> tuple[str, str]:
    """
    Normalize edi and release to the zero-padded strings Stedi expects.

    Examples
    --------
    _normalize(315,  4010)    → ("315",    "004010")
    _normalize("315", "4010") → ("315",    "004010")
    _normalize("315", "004010") → ("315",  "004010")
    _normalize(856,   5010)   → ("856",    "005010")
    """
    edi_str     = str(edi).strip()
    release_str = str(release).strip().zfill(6)   # "4010" → "004010"
    return edi_str, release_str

def _tx_url(edi: str, release: str) -> str:
    """https://www.stedi.com/edi/x12-004010/315"""
    return f"{BASE}/edi/x12-{release}/{edi}"

def _seg_url(seg: str, release: str) -> str:
    """https://www.stedi.com/edi/x12-004010/segment/B4"""
    return f"{BASE}/edi/x12-{release}/segment/{seg}"


# ── Data classes ──────────────────────────────────────────────────────────────

@dataclass
class TxSegmentRow:
    """One row in a transaction-set segment table."""
    position:     str = ""
    segment_id:   str = ""
    segment_name: str = ""
    requirement:  str = ""   # Mandatory | Optional | Conditional
    max_use:      str = ""
    loop:         str = ""   # populated for segments nested inside a loop
    description:  str = ""

@dataclass
class ElementRow:
    """One element within a segment definition."""
    segment_id:   str = ""
    position:     str = ""   # e.g. B4-01
    element_id:   str = ""   # e.g. 152
    element_name: str = ""
    data_type:    str = ""   # ID | AN | DT | TM | N0 | R …
    requirement:  str = ""   # Mandatory | Optional | Conditional
    min_len:      str = ""
    max_len:      str = ""
    repeat:       str = ""
    description:  str = ""
    notes:        str = ""   # conditional rule text

@dataclass
class SegmentSpec:
    segment_id:   str = ""
    segment_name: str = ""
    purpose:      str = ""
    url:          str = ""
    elements:     list[ElementRow] = field(default_factory=list)

@dataclass
class TransactionSetSpec:
    """Complete spec for one transaction set."""
    edi:          str = ""   # e.g. "315"
    release:      str = ""   # e.g. "004010"
    name:         str = ""
    description:  str = ""
    url:          str = ""
    segments:     list[TxSegmentRow] = field(default_factory=list)


# ── HTTP ──────────────────────────────────────────────────────────────────────

def _fetch(url: str) -> BeautifulSoup:
    print(f"  GET {url}")
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    time.sleep(SLEEP)
    return BeautifulSoup(r.text, "html.parser")

def _clean(node) -> str:
    text = node.get_text() if hasattr(node, "get_text") else str(node)
    return re.sub(r"\s+", " ", text).strip()


# ── fetch_transaction_set ────────────────────────────────────────────────────
#
# Parses the transaction-set segment table from Stedi.
#
# Stedi renders each segment as a line in the page text like:
#   "010STTransaction Set HeaderMandatory->Max 1"   (004010 uses 3-digit pos)
#   "0100STTransaction Set HeaderMandatory->Max 1"  (newer releases use 4-digit)
#
# Segment IDs are ALL-CAPS/digits, immediately followed by a TitleCase name.
# We split using a lookahead: the name always starts Uppercase + lowercase.
#
# Loop boundaries appear as their own line, e.g.:
#   "R4 Loop Mandatory" / "R4 Loop Mandatory Repeat 20"

_TX_ROW_RE = re.compile(
    r"(\d{2,4})"           # 2–4 digit position
    r"([A-Z][A-Z0-9]+?)"  # segment ID — shortest all-caps match
    r"(?=[A-Z][a-z])"     # lookahead: name starts Uppercase+lowercase
    r"(.+?)"              # segment name (non-greedy)
    r"\s*(Mandatory|Optional|Conditional)"
    r".*?->Max\s+(\d+)"
)

_LOOP_RE = re.compile(
    r"([A-Z][A-Z0-9]{1,3})\s+Loop\s+(Mandatory|Optional)"
)


def fetch_transaction_set(edi: "str | int", release: "str | int" = "004010") -> TransactionSetSpec:
    """
    Fetch and parse the transaction-set segment table for any X12 EDI/release.

    Parameters
    ----------
    edi     : Transaction set number — string or int. e.g. "315" or 315
    release : X12 release — string or int. e.g. "004010", "4010", or 4010

    Returns
    -------
    TransactionSetSpec with .segments populated as list[TxSegmentRow]
    """
    edi, release = _normalize(edi, release)
    url  = _tx_url(edi, release)
    soup = _fetch(url)

    spec = TransactionSetSpec(edi=edi, release=release, url=url)

    # Name and description from headings
    h1 = soup.find("h1")
    if h1:
        spec.name = _clean(h1).removeprefix(f"EDI {edi} ").strip()

    # The purpose paragraph follows the headings
    for h2 in soup.find_all("h2"):
        text = _clean(h2)
        # Skip short functional/subcommittee headers
        if len(text) > 40:
            spec.description = text
            break

    # Build a description map from <li> anchor elements
    desc_map: dict[str, str] = {}
    for li in soup.find_all("li"):
        a = li.find("a")
        if not a:
            continue
        link_text = _clean(a)
        m = re.match(r"\d+([A-Z][A-Z0-9]+)", link_text)
        if m:
            seg_id   = m.group(1)
            li_text  = _clean(li)
            a_text   = _clean(a)
            remainder = li_text.replace(a_text, "").strip()
            if remainder:
                desc_map[seg_id] = remainder

    # Parse segment rows from the page text
    rows: list[TxSegmentRow] = []
    current_loop = ""

    for raw_line in soup.get_text(separator="\n").splitlines():
        line = re.sub(r"\s+", " ", raw_line).strip()
        if not line:
            continue

        lm = _LOOP_RE.search(line)
        if lm:
            current_loop = lm.group(1)
            continue

        m = _TX_ROW_RE.search(line)
        if m:
            seg_id = m.group(2)
            rows.append(TxSegmentRow(
                position    = m.group(1),
                segment_id  = seg_id,
                segment_name= m.group(3).strip(),
                requirement = m.group(4),
                max_use     = m.group(5),
                loop        = current_loop,
                description = desc_map.get(seg_id, ""),
            ))

    # Deduplicate on (position, segment_id) preserving order
    seen: set[tuple] = set()
    for r in rows:
        key = (r.position, r.segment_id)
        if key not in seen:
            seen.add(key)
            spec.segments.append(r)

    print(f"     → {len(spec.segments)} segment rows for EDI {edi} ({release})")
    return spec


# ── extract_unique_segments ──────────────────────────────────────────────────

def extract_unique_segments(edi_path: Optional[str],
                            default: Optional[list[str]] = None) -> list[str]:
    """
    Read an EDI file and return sorted unique segment IDs.

    Parameters
    ----------
    edi_path : Path to an EDI file.  May contain literal \\x15 or escaped \\\\x15.
    default  : Fallback list if no file is provided.
    """
    if not edi_path:
        fallback = default or ["ISA","GS","ST","B4","N9","Q2","R4","DTM","SE","GE","IEA"]
        print(f"\n  No EDI file — using default segments: {fallback}")
        return fallback

    print(f"\n  Extracting unique segment IDs from: {edi_path}")
    raw  = Path(edi_path).read_text(encoding="utf-8")
    norm = (raw
            .replace("\\x15", "\x15")
            .replace("\r\n", "").replace("\n", "").replace("\r", ""))

    segs = [s.strip() for s in norm.split("\x15") if s.strip()]
    ids  = sorted(set(s.split("*")[0] for s in segs if s))
    print(f"     → {len(ids)} unique IDs: {ids}")
    return ids


# ── fetch_segment_spec ───────────────────────────────────────────────────────
#
# Parses the element table for one segment from Stedi.
# Confirmed element row format from live pages (N9, R4, DTM, B4):
#
#   "N9-01  128  Reference Identification Qualifier  Identifier (ID)  Mandatory  2  3  -"
#
# Composite elements have no min/max:
#   "N9-07  C040  Reference Identifier  Composite (composite)  Optional"

def _build_elem_re(seg_id: str) -> re.Pattern:
    s = re.escape(seg_id)
    return re.compile(
        rf"({s}-(\d{{2}}))"                                              # position e.g. N9-01
        r"\s+(\w+)"                                                      # element ID (numeric or Cxxx composite)
        r"\s+(.+?)"                                                      # element name
        r"\s+(Identifier|String|Numeric|Date|Time|Decimal|Binary|Composite)"
        r"\s*\((\w+)\)"                                                  # type code
        r"\s+(Mandatory|Optional|Conditional)"
        r"(?:\s+(\d+)\s+(\d+))?"                                        # min max (absent for composites)
        r"\s*([-\d]+)?"                                                  # repeat
    )


def fetch_segment_spec(seg_id: str, release: "str | int" = "004010") -> SegmentSpec:
    """
    Fetch and parse the element table for one X12 segment.

    Parameters
    ----------
    seg_id  : Segment identifier, e.g. "B4", "N9", "DTM"
    release : X12 release — string or int. e.g. "004010", "4010", or 4010

    Returns
    -------
    SegmentSpec with .elements populated as list[ElementRow]
    """
    _, release = _normalize(seg_id, release)  # normalize release only
    url  = _seg_url(seg_id, release)
    soup = _fetch(url)
    spec = SegmentSpec(segment_id=seg_id, url=url)

    h1 = soup.find("h1")
    if h1:
        spec.segment_name = _clean(h1).removeprefix(f"{seg_id} ").strip()
    h2s = soup.find_all("h2")
    if h2s:
        spec.purpose = _clean(h2s[0])

    full = soup.get_text(separator="\n")
    elements: list[ElementRow] = []

    for m in _build_elem_re(seg_id).finditer(full):
        elements.append(ElementRow(
            segment_id  = seg_id,
            position    = m.group(1),
            element_id  = m.group(3),
            element_name= m.group(4).strip(),
            data_type   = m.group(6),
            requirement = m.group(7),
            min_len     = m.group(8) or "",
            max_len     = m.group(9) or "",
            repeat      = (m.group(10) or "").strip(),
        ))

    # "XXnn is …" notes  e.g. "B404 is the date of last reported status"
    note_re = re.compile(
        rf"{re.escape(seg_id)}(\d{{2}})\s+(?:is|reflects)\s+(.+?)\.(?=\s|$)",
        re.DOTALL
    )
    notes_map: dict[str, str] = {}
    for nm in note_re.finditer(full):
        notes_map[f"{seg_id}-{nm.group(1)}"] = re.sub(r"\s+", " ", nm.group(2)).strip()

    # Conditional/relational rules  e.g. "C0403: If DTM-04 is present …"
    cond_re   = re.compile(r"[CPOR]\d{4}:\s*(.+?)(?=\n[CPOR]\d{4}:|\Z)", re.DOTALL)
    cond_rules = [re.sub(r"\s+", " ", r).strip() for r in cond_re.findall(full)]

    for el in elements:
        if el.position in notes_map:
            el.notes = notes_map[el.position]
        pos_num = el.position.split("-")[1] if "-" in el.position else ""
        for rule in cond_rules:
            if pos_num and f"-{pos_num}" in rule and rule not in (el.notes or ""):
                el.notes = ((el.notes + " | " + rule) if el.notes else rule).strip(" | ")

    # Fallback: <table> rows
    if not elements:
        for tbl in soup.find_all("table"):
            for tr in tbl.find_all("tr")[1:]:
                cells = [_clean(td) for td in tr.find_all(["td","th"])]
                if len(cells) >= 5 and cells[0]:
                    elements.append(ElementRow(
                        segment_id=seg_id, position=cells[0],
                        element_id=cells[1] if len(cells)>1 else "",
                        element_name=cells[2] if len(cells)>2 else "",
                        data_type=cells[3] if len(cells)>3 else "",
                        requirement=cells[4] if len(cells)>4 else "",
                        min_len=cells[5] if len(cells)>5 else "",
                        max_len=cells[6] if len(cells)>6 else "",
                        repeat=cells[7] if len(cells)>7 else "",
                    ))

    spec.elements = elements
    print(f"     {seg_id:6}: {len(elements):3} elements   ({spec.segment_name})")
    return spec


# ── Save helpers ──────────────────────────────────────────────────────────────

def save_json(data: object, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=str)
    print(f"       → {path}")

def save_csv(rows: list[dict], path: str) -> None:
    if not rows:
        return
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)
    print(f"       → {path}")


# ── run() — works in notebooks, scripts, and imports ─────────────────────────

def run(
    edi:      "str | int" = "315",
    release:  "str | int" = "004010",
    edi_file: Optional[str] = None,
) -> None:
    """
    Scrape the transaction-set table and all segment specs for one EDI/release,
    then save five output files to the current directory.

    Works from a notebook, a script, or a plain import.

    Parameters
    ----------
    edi      : Transaction set number — string or int. e.g. "315" or 315
    release  : X12 release — string or int. e.g. "004010", "4010", or 4010
    edi_file : Path to a raw EDI file used to derive the unique segment list.
               Pass None to use the built-in defaults for the chosen EDI.

    Examples
    --------
    # notebook
    from scrape_315_spec import run
    run(edi=315, release=4010)
    run(edi="856", release="005010", edi_file="myshipment.edi")

    # command line (via __main__ below)
    python scrape_315_spec.py --edi 315 --release 4010
    python scrape_315_spec.py myfile.edi --edi 856 --release 5010
    """
    edi, release = _normalize(edi, release)
    prefix = f"{edi}_{release}"

    print(f"\nScraping EDI {edi} / X12-{release} from stedi.com …")

    # 1 ── transaction-set table
    print("\n[1] Transaction-set table")
    tx_spec  = fetch_transaction_set(edi=edi, release=release)
    tx_dicts = [asdict(r) for r in tx_spec.segments]
    save_json(tx_dicts, f"{prefix}_transaction_table.json")
    save_csv(tx_dicts,  f"{prefix}_transaction_table.csv")

    # 2 ── unique segments
    print("\n[2] Unique segments")
    unique_segs = extract_unique_segments(edi_file)
    save_json(
        {"edi": edi, "release": release,
         "edi_file": edi_file or "(default)",
         "unique_segments": unique_segs, "count": len(unique_segs)},
        f"{prefix}_unique_segments.json",
    )

    # 3 ── segment element tables
    print(f"\n[3] Fetching element tables for {len(unique_segs)} segments …")
    all_specs:    list[SegmentSpec] = []
    all_elements: list[dict]        = []

    for seg in unique_segs:
        try:
            spec = fetch_segment_spec(seg_id=seg, release=release)
        except requests.HTTPError as e:
            print(f"     SKIP {seg}: HTTP {e.response.status_code}")
            spec = SegmentSpec(segment_id=seg, segment_name=f"(HTTP {e.response.status_code})")
        except Exception as e:
            print(f"     ERROR {seg}: {e}")
            spec = SegmentSpec(segment_id=seg, segment_name="(error)", purpose=str(e))
        all_specs.append(spec)
        all_elements.extend(asdict(el) for el in spec.elements)

    # 4 ── save
    print("\n[4] Saving segment specs …")
    save_json([asdict(s) for s in all_specs], f"{prefix}_segment_specs.json")
    save_csv(all_elements,                     f"{prefix}_segment_specs_flat.csv")

    print(f"""
╔══════════════════════════════════════════════╗
║           Scrape Complete  ✓                 ║
╚══════════════════════════════════════════════╝
  EDI / Release        : {edi} / {release}
  Transaction rows     : {len(tx_dicts)}
  Unique segments      : {len(unique_segs)}
  Total elements       : {len(all_elements)}

  Output files  (prefix: {prefix}_)
  ─────────────────────────────────
  {prefix}_transaction_table.json / .csv
  {prefix}_unique_segments.json
  {prefix}_segment_specs.json
  {prefix}_segment_specs_flat.csv
""")


# ── CLI entry point (never runs in a notebook) ────────────────────────────────

#if __name__ == "__main__":
#    import argparse
#    parser = argparse.ArgumentParser(description="Scrape X12 EDI specs from stedi.com")
#    parser.add_argument("edi_file",  nargs="?",         help="Path to a raw EDI file (optional)")
#    parser.add_argument("--edi",     default="315",     help="Transaction set number (default: 315)")
#    parser.add_argument("--release", default="004010",  help="X12 release (default: 004010)")
#    args = parser.parse_args()
#    run(edi=args.edi, release=args.release, edi_file=args.edi_file)


In [28]:
run(edi='214', release='4010')


Scraping EDI 214 / X12-004010 from stedi.com …

[1] Transaction-set table
  GET https://www.stedi.com/edi/x12-004010/214
     → 0 segment rows for EDI 214 (004010)
       → 214_004010_transaction_table.json

[2] Unique segments

  No EDI file — using default segments: ['ISA', 'GS', 'ST', 'B4', 'N9', 'Q2', 'R4', 'DTM', 'SE', 'GE', 'IEA']
       → 214_004010_unique_segments.json

[3] Fetching element tables for 11 segments …
  GET https://www.stedi.com/edi/x12-004010/segment/ISA
     ISA   :  15 elements   (Interchange Control Header)
  GET https://www.stedi.com/edi/x12-004010/segment/GS
     GS    :   8 elements   (Functional Group Header)
  GET https://www.stedi.com/edi/x12-004010/segment/ST
     ST    :   2 elements   (Transaction Set Header)
  GET https://www.stedi.com/edi/x12-004010/segment/B4
     B4    :  13 elements   (Beginning Segment for Inquiry or Reply)
  GET https://www.stedi.com/edi/x12-004010/segment/N9
     N9    :   7 elements   (Reference Identification)
  GET https:/